In [ ]:
import os
import boto3

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID", "")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY", "")
IMAGE_PATH = r"google_test_images\adversarial\img_0000.png"

rekognition = boto3.client(
    "rekognition",
    region_name="us-east-1",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY
)

with open(IMAGE_PATH, "rb") as f:
    image_bytes = f.read()

response = rekognition.detect_labels(
    Image={"Bytes": image_bytes},
    MaxLabels=10,
    MinConfidence=0
)

print("\nAWS Rekognition predictions:\n")

for i, label in enumerate(response["Labels"], start=1):
    print(
        f"{i:2d}. {label['Name']:25s} "
        f"{label['Confidence']:.2f}%"
    )

In [ ]:
import os
import boto3
import pandas as pd
from tqdm import tqdm


# ============================================================
# CONFIG
# ============================================================

REGION = "us-east-1"

IMAGE_FOLDER = r"google_test_images\SMER_ViT"
OUTPUT_CSV = "aws_adversarial_results_SMER_ViT.csv"

# IMAGE_FOLDER = r"google_test_images\benign"
# OUTPUT_CSV = "aws_benign_results.csv"

TOP_K = 10


# ============================================================
# AWS CLIENT
# ============================================================

rekognition = boto3.client(
    "rekognition",
    region_name=REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY
)


# ============================================================
# PREDICTION FUNCTION
# ============================================================

def get_aws_labels(image_path):

    with open(image_path, "rb") as f:
        image_bytes = f.read()

    response = rekognition.detect_labels(
        Image={"Bytes": image_bytes},
        MaxLabels=TOP_K,
        MinConfidence=0
    )

    return response["Labels"]


# ============================================================
# FIND IMAGES
# ============================================================

extensions = (
    ".png",
    ".jpg",
    ".jpeg",
    ".webp"
)

images = sorted([
    f for f in os.listdir(IMAGE_FOLDER)
    if f.lower().endswith(extensions)
])

# imagga = pd.read_csv("imagga_adversarial_results.csv")

# images = imagga["image"].tolist()

print("Number of images:", len(images)) 


print("=" * 60)
print("AWS REKOGNITION TEST")
print("=" * 60)
print("Folder:", IMAGE_FOLDER)
print("Number of images:", len(images))
print("=" * 60)


# ============================================================
# PROCESS
# ============================================================

rows = []

for filename in tqdm(images):

    image_path = os.path.join(
        IMAGE_FOLDER,
        filename
    )

    row = {
        "image": filename
    }

    try:

        labels = get_aws_labels(image_path)

        row["status"] = "success"

        for i, label in enumerate(
            labels[:TOP_K],
            start=1
        ):

            row[f"top{i}_label"] = label["Name"]
            row[f"top{i}_confidence"] = label["Confidence"]

    except Exception as e:

        row["status"] = "failed"
        row["error"] = str(e)

    rows.append(row)


# ============================================================
# SAVE CSV
# ============================================================

df = pd.DataFrame(rows)

df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nFinished.")

print("Saved:")
print(OUTPUT_CSV)

print(
    "\nSuccessful:",
    (df["status"] == "success").sum()
)

print(
    "Failed:",
    (df["status"] == "failed").sum()
)

In [ ]:
import pandas as pd

BENIGN_CSV = "aws_benign_results.csv"
ADV_CSV = "aws_adversarial_results_SMER_ViT.csv"

TOP_K = 5

benign = pd.read_csv(BENIGN_CSV)
adv = pd.read_csv(ADV_CSV)

# Keep only successful queries
benign = benign[benign["status"] == "success"].copy()
adv = adv[adv["status"] == "success"].copy()

# Merge same images
df = benign.merge(
    adv,
    on="image",
    suffixes=("_benign", "_adv")
)

print("Paired images:", len(df))


# ============================================================
# 1. TOP-1 LABEL CHANGE
# ============================================================

df["top1_changed"] = (
    df["top1_label_benign"].str.lower()
    !=
    df["top1_label_adv"].str.lower()
)

top1_change_rate = df["top1_changed"].mean() * 100


# ============================================================
# 2. TOP-1 CONFIDENCE DROP
# ============================================================

df["top1_conf_drop"] = (
    df["top1_confidence_benign"]
    -
    df["top1_confidence_adv"]
)

mean_conf_drop = df["top1_conf_drop"].mean()


# ============================================================
# 3. TOP-K LABEL OVERLAP
# ============================================================

def get_topk(row, suffix, k):

    labels = []

    for i in range(1, k + 1):

        col = f"top{i}_label_{suffix}"

        if col in row and pd.notna(row[col]):
            labels.append(
                str(row[col]).strip().lower()
            )

    return set(labels)


def overlap(row):

    benign_labels = get_topk(
        row,
        "benign",
        TOP_K
    )

    adv_labels = get_topk(
        row,
        "adv",
        TOP_K
    )

    if len(benign_labels) == 0:
        return 0

    return len(
        benign_labels & adv_labels
    ) / len(benign_labels)


df[f"top{TOP_K}_overlap"] = df.apply(
    overlap,
    axis=1
)

mean_overlap = (
    df[f"top{TOP_K}_overlap"].mean()
    * 100
)


# ============================================================
# 4. COMPLETE TOP-K DISRUPTION
# ============================================================

df["complete_disruption"] = (
    df[f"top{TOP_K}_overlap"] == 0
)

complete_disruption_rate = (
    df["complete_disruption"].mean()
    * 100
)


# ============================================================
# PRINT RESULTS
# ============================================================

print("\n====================================")
print("AWS ADVERSARIAL EVALUATION")
print("====================================")

print(
    f"Number of paired images: {len(df)}"
)

print(
    f"Top-1 Label Change Rate: "
    f"{top1_change_rate:.2f}%"
)

print(
    f"Mean Top-1 Confidence Drop: "
    f"{mean_conf_drop:.2f}"
)

print(
    f"Top-{TOP_K} Label Retention: "
    f"{mean_overlap:.2f}%"
)

print(
    f"Top-{TOP_K} Complete Disruption: "
    f"{complete_disruption_rate:.2f}%"
)


# ============================================================
# SAVE DETAILED RESULTS
# ============================================================

df.to_csv(
    "aws_pairwise_evaluation.csv",
    index=False
)

print(
    "\nSaved: aws_pairwise_evaluation.csv"
)

In [ ]:
import pandas as pd
import re


# ============================================================
# FILES
# ============================================================

LABELS_CSV = "google_test_images/labels.csv"

BENIGN_CSV = "aws_benign_results.csv"
ADV_CSV = "aws_adversarial_results_SMER_ViT.csv"

OUTPUT_CSV = "aws_evaluation_detailed.csv"


# ============================================================
# LOAD FILES
# ============================================================

labels = pd.read_csv(LABELS_CSV)
benign = pd.read_csv(BENIGN_CSV)
adv = pd.read_csv(ADV_CSV)


# Only successful API requests
if "status" in benign.columns:
    benign = benign[benign["status"] == "success"].copy()

if "status" in adv.columns:
    adv = adv[adv["status"] == "success"].copy()


# ============================================================
# MERGE BENIGN + ADVERSARIAL + GROUND TRUTH
# ============================================================

df = labels.merge(
    benign,
    left_on="benign_image",
    right_on="image",
    how="inner"
)

df = df.merge(
    adv,
    left_on="adversarial_image",
    right_on="image",
    how="inner",
    suffixes=("_benign", "_adv")
)

print("Number of paired images:", len(df))


# ============================================================
# NORMALIZE TEXT
# ============================================================

def normalize(text):

    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    text = text.replace("-", " ")
    text = text.replace("_", " ")

    text = re.sub(
        r"[^a-z0-9 ]",
        "",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text


# ============================================================
# SEMANTIC MATCH
# ============================================================
#
# Examples:
#
# hammerhead <-> hammerhead shark
# tree frog   <-> frog
# king snake  <-> snake
#
# This is intentionally more permissive than exact matching
# because AWS returns general concepts rather than ImageNet
# class names.
# ============================================================

def semantic_match(gt, prediction):

    gt = normalize(gt)
    prediction = normalize(prediction)

    if not gt or not prediction:
        return False

    # Exact
    if gt == prediction:
        return True

    # Phrase contained
    if gt in prediction or prediction in gt:
        return True

    # Shared word
    gt_words = set(gt.split())
    pred_words = set(prediction.split())

    common = gt_words.intersection(pred_words)

    if len(common) > 0:
        return True

    return False


# ============================================================
# CORRECT AT K
# ============================================================

def correct_at_k(row, suffix, k):

    gt = row["label_name"]

    for i in range(1, k + 1):

        col = f"top{i}_label_{suffix}"

        if col not in row.index:
            continue

        prediction = row[col]

        if semantic_match(
            gt,
            prediction
        ):
            return True

    return False


# ============================================================
# BENIGN + ADV CORRECTNESS
# ============================================================

for k in [1, 5, 10]:

    df[f"benign_top{k}_correct"] = df.apply(
        lambda row: correct_at_k(
            row,
            "benign",
            k
        ),
        axis=1
    )

    df[f"adv_top{k}_correct"] = df.apply(
        lambda row: correct_at_k(
            row,
            "adv",
            k
        ),
        axis=1
    )


# ============================================================
# ACCURACY
# ============================================================

print("\n============================================")
print("        AWS REKOGNITION ROBUSTNESS")
print("============================================")

summary = {}

for k in [1, 5, 10]:

    benign_acc = (
        df[f"benign_top{k}_correct"].mean()
        * 100
    )

    adv_acc = (
        df[f"adv_top{k}_correct"].mean()
        * 100
    )

    accuracy_drop = (
        benign_acc - adv_acc
    )

    summary[f"benign_top{k}"] = benign_acc
    summary[f"adv_top{k}"] = adv_acc
    summary[f"drop_top{k}"] = accuracy_drop

    print(f"\nTOP-{k}")

    print(
        f"Benign accuracy:       "
        f"{benign_acc:.2f}%"
    )

    print(
        f"Adversarial accuracy:  "
        f"{adv_acc:.2f}%"
    )

    print(
        f"Accuracy drop:         "
        f"{accuracy_drop:.2f} pp"
    )


# ============================================================
# ATTACK SUCCESS RATE
# ============================================================
#
# ASR =
#
# benign correct -> adversarial incorrect
# ----------------------------------------
#       benign correct
#
# ============================================================

print("\n============================================")
print("             ATTACK SUCCESS RATE")
print("============================================")

for k in [1, 5, 10]:

    benign_correct = df[
        df[f"benign_top{k}_correct"]
    ]

    successful_attacks = benign_correct[
        ~benign_correct[
            f"adv_top{k}_correct"
        ]
    ]

    if len(benign_correct) > 0:

        asr = (
            len(successful_attacks)
            /
            len(benign_correct)
            * 100
        )

    else:
        asr = 0

    summary[f"asr_top{k}"] = asr

    print(f"\nTop-{k}:")

    print(
        "Benign correct:",
        len(benign_correct)
    )

    print(
        "Successfully attacked:",
        len(successful_attacks)
    )

    print(
        f"ASR: {asr:.2f}%"
    )


# ============================================================
# TOP-1 LABEL CHANGE
# ============================================================

df["top1_changed"] = (

    df["top1_label_benign"]
    .fillna("")
    .apply(normalize)

    !=

    df["top1_label_adv"]
    .fillna("")
    .apply(normalize)
)

top1_change_rate = (
    df["top1_changed"].mean()
    * 100
)

summary["top1_change"] = (
    top1_change_rate
)


# ============================================================
# TOP-1 CONFIDENCE DROP
# ============================================================

df["top1_confidence_drop"] = (

    df["top1_confidence_benign"]

    -

    df["top1_confidence_adv"]
)

mean_confidence_drop = (
    df["top1_confidence_drop"]
    .mean()
)

summary["confidence_drop"] = (
    mean_confidence_drop
)


# ============================================================
# TOP-5 LABEL RETENTION
# ============================================================

def get_topk_labels(row, suffix, k):

    labels_list = []

    for i in range(1, k + 1):

        col = f"top{i}_label_{suffix}"

        if col not in row.index:
            continue

        value = row[col]

        if pd.notna(value):

            labels_list.append(
                normalize(value)
            )

    return set(labels_list)


def top5_overlap(row):

    benign_labels = get_topk_labels(
        row,
        "benign",
        5
    )

    adv_labels = get_topk_labels(
        row,
        "adv",
        5
    )

    if len(benign_labels) == 0:
        return 0

    overlap = (
        len(
            benign_labels.intersection(
                adv_labels
            )
        )
        /
        len(benign_labels)
    )

    return overlap


df["top5_retention"] = df.apply(
    top5_overlap,
    axis=1
)

top5_retention = (
    df["top5_retention"].mean()
    * 100
)

summary["top5_retention"] = (
    top5_retention
)


# ============================================================
# COMPLETE TOP-5 DISRUPTION
# ============================================================

df["top5_complete_disruption"] = (
    df["top5_retention"] == 0
)

complete_disruption = (
    df[
        "top5_complete_disruption"
    ].mean()
    * 100
)

summary["complete_disruption"] = (
    complete_disruption
)


# ============================================================
# PRINT EXTRA METRICS
# ============================================================

print("\n============================================")
print("          PREDICTION DISRUPTION")
print("============================================")

print(
    f"Top-1 Label Change Rate: "
    f"{top1_change_rate:.2f}%"
)

print(
    f"Mean Top-1 Confidence Drop: "
    f"{mean_confidence_drop:.2f} pp"
)

print(
    f"Top-5 Label Retention: "
    f"{top5_retention:.2f}%"
)

print(
    f"Top-5 Complete Disruption: "
    f"{complete_disruption:.2f}%"
)


# ============================================================
# SAVE DETAILED RESULTS
# ============================================================

df.to_csv(
    OUTPUT_CSV,
    index=False
)

print(
    "\nDetailed results saved:",
    OUTPUT_CSV
)


# ============================================================
# PAPER-STYLE SUMMARY TABLE
# ============================================================

summary_table = pd.DataFrame({

    "Service": [
        "AWS Rekognition"
    ],

    "N": [
        len(df)
    ],

    "Benign Top-1": [
        summary["benign_top1"]
    ],

    "Adv Top-1": [
        summary["adv_top1"]
    ],

    "Top-1 ASR": [
        summary["asr_top1"]
    ],

    "Benign Top-5": [
        summary["benign_top5"]
    ],

    "Adv Top-5": [
        summary["adv_top5"]
    ],

    "Top-5 ASR": [
        summary["asr_top5"]
    ],

    "Benign Top-10": [
        summary["benign_top10"]
    ],

    "Adv Top-10": [
        summary["adv_top10"]
    ],

    "Top-10 ASR": [
        summary["asr_top10"]
    ],

    "Prediction Change": [
        summary["top1_change"]
    ]
})


# Round
numeric_columns = (
    summary_table
    .select_dtypes(
        include="number"
    )
    .columns
)

summary_table[numeric_columns] = (
    summary_table[
        numeric_columns
    ].round(2)
)


print("\n============================================")
print("              FINAL TABLE")
print("============================================")

print(
    summary_table.to_string(
        index=False
    )
)

summary_table.to_csv(
    "aws_summary.csv",
    index=False
)

In [ ]:
summary_table